# Isaac Sim Jupyter Notebook Tutorial

This notebook will demonstrate step by step how to load the IAI apartment a robot into the simulation environment.
 
> Select the kernel `Python 3.11.13 /mnt/dev-tools/isaac-sim-5.1/python.sh` if you are running in VScode.
>
> "Control + Enter" to execute the selected code cell. 

<!-- <button data-commandlinker-command="notebook:restart" class="jupyter-button">Force Stop</button> -->

## Start the GPU monitor and virtual desktop

Make sure it has at least 3000M free memory.

> Note: We will enable the ROS bridge extension in later step, so you can ignore the warning about ROS not being installed.

In [1]:
from gpu_monitor import GPUMonitor
# Monitor GPU usage
gpu_monitor = GPUMonitor()

from utils import *
# Extract precompiled cache
run_script(CACHE_EXTRACT_CMD)
# Open Desktop in sidecar
display_desktop()

HTML(value='')

rclpy not installed! Have you source ROS2 environment?


HTML(value='<a href="https://jupyter.dev.intel4coro.de//user/yxzhan-isaacsim-template-dj0wlqco/desktop"  class…

## Start SimulationApp

The application window is frozen and non-interactive, which is normal.

<div style="color:red">This will take some time, so give it a minute and wait for it to say <b>"SimulationApp Ready!"</b> before you go to next step.</div>

In [2]:
from isaacsim import SimulationApp
from IPython.display import clear_output
import sys
import builtins

original_stdout = sys.stdout
original_stderr = sys.stderr

simulation_app = SimulationApp({
    "headless": False,
    # "hide_ui": True,
    "width": 1280,
    "height": 960,
    "renderer": "RaytracedLighting",
    "display_options": 3286,  # Setsimulation_app.update() display options to show default grid
})

# Fix the issue where notebook output is being hijacked by Isaac Sim.
sys.stdout = original_stdout
sys.stderr = original_stderr
clear_output(wait=True)
print('SimulationApp Ready!')

SimulationApp Ready!


## Define the physical properties of the simulation environment

In [3]:
from isaacsim.core.api import World

my_world = World(stage_units_in_meters=1.0,
                 physics_dt=1 / 200,
                 rendering_dt=8 / 200)
my_world.reset()

## Refresh View

Until now, we don't see any change in the app window, that's because it needs to be manually refreshed.

In [4]:
from tqdm import tqdm

def refresh_view(steps=10):
    bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} steps]"
    for i in tqdm(range(steps), desc="World Stepping", ncols=60, bar_format=bar_format):
        my_world.step(render=True)

refresh_view()

World Stepping: 100%|█████████████████████████| 10/10 steps]


## Spawn environment USD to the world

The apartment USD is converted from the [iai apartment URDF](https://github.com/code-iai/iai_maps/blob/ros-jazzy/iai_apartment/urdf/apartment.urdf).

[Tutorial: import URDF](https://docs.isaacsim.omniverse.nvidia.com/5.1.0/importer_exporter/import_urdf.html#isaac-sim-app-tutorial-advanced-import-urdf)

In [5]:
import os
from isaacsim.core.utils.prims import define_prim

prim = define_prim("/World/Apartment", "Xform")
asset_path = f"{os.getcwd()}/../usd/apartment/apartment.usd"
prim.GetReferences().AddReference(asset_path)

refresh_view()

World Stepping: 100%|█████████████████████████| 10/10 steps]


## Change camera position

In [6]:
import numpy as np
from isaacsim.core.utils import viewports

viewports.set_camera_view(eye=np.array([5, 0, 3]), target=np.array([0, 0, 0]))
refresh_view()

World Stepping: 100%|█████████████████████████| 10/10 steps]


## Spawn Robot Anymal

The USD file for the Anymal robot is provided by Isaac Sim itself. 

You will see the robot collapse on the ground because no control commands have been sent to it yet.

In [7]:
from isaacsim.robot.policy.examples.robots import AnymalFlatTerrainPolicy

robot = AnymalFlatTerrainPolicy(
    prim_path="/World/Anymal",
    name="Anymal",
    position=np.array([0, 0, 0.6]),
)
refresh_view(100)

World Stepping: 100%|███████████████████████| 100/100 steps]


## Add physics callback function to control the robot

The control command is a 3-element array, where the first value represents forward velocity, the second represents lateral (left/right) movement, and the third represents rotation. Value range is from -1 to 1.

In [8]:
first_step = True
commands = [0.0, 0.0, 0.0]

def on_physics_step(step_size) -> None:
    global first_step, commands
    if first_step:
        robot.initialize()
        first_step = False
    else:
        robot.forward(step_size, commands)

my_world.add_physics_callback("physics_step", callback_fn=on_physics_step)

refresh_view(100)

World Stepping: 100%|███████████████████████| 100/100 steps]


## Restart Simulation and initialize robot

In [9]:
first_step = True
my_world.reset()
commands = [0.0, 0.0, 0.0]
refresh_view(100)

World Stepping: 100%|███████████████████████| 100/100 steps]


## Send Control Commands

In [10]:
# Forward
commands = [0.3, 0.0, 0.0]
refresh_view(steps=100)
# Move left
commands = [0.0, 0.3, 0.0]
refresh_view(steps=100)
# Turn around
commands = [0.0, 0.0, 0.8]
refresh_view(steps=100)
# Stop
commands = [0.0, 0.0, 0.0]
refresh_view(steps=100)

World Stepping: 100%|███████████████████████| 100/100 steps]
World Stepping: 100%|███████████████████████| 100/100 steps]
World Stepping: 100%|███████████████████████| 100/100 steps]
World Stepping: 100%|███████████████████████| 100/100 steps]


## Spawn Kitchen Objects

Load USD files from the repo

In [11]:
from isaacsim.core.utils.prims import create_prim
from pxr import Gf
import random
import string

object_list = [
    "stretch",
    "Table049",
    "Toaster003",
]

for i in range(len(object_list)):
    obj = object_list[i]
    obj_prim = f"/World/{obj}"
    create_prim(
        usd_path=f"{os.getcwd()}/../usd/{obj}/{obj}.usd",
        prim_path=obj_prim,
        translation=Gf.Vec3d(2, -1 + i, 0.2)
    )

refresh_view(steps=100)

World Stepping: 100%|███████████████████████| 100/100 steps]


## Delete Kitchen Objects

In [12]:
from isaacsim.core.utils.prims import delete_prim

for obj in object_list:
    delete_prim( f"/World/{obj}")

refresh_view()

World Stepping: 100%|█████████████████████████| 10/10 steps]


## Enable ROS Bridge 

In [16]:
!env

SHELL=/bin/bash
ROS_VERSION=2
KUBERNETES_SERVICE_PORT_HTTPS=443
NVIDIA_VISIBLE_DEVICES=GPU-c668e0e1-a8fa-896c-91c3-2a52f127bc2c
BINDER_REQUEST=v2/gh/yxzhan/isaacsim-template/main
JUPYTERHUB_ADMIN_ACCESS=1
KUBERNETES_SERVICE_PORT=443
USDIMAGING_ENABLE_SPARSE_LIGHT_UPDATES=1
PROXY_API_SERVICE_HOST=10.152.183.141
VULKAN_HEADERS_INSTALL_DIR=
JUPYTERHUB_SERVICE_URL=http://0.0.0.0:8888/user/yxzhan-isaacsim-template-dj0wlqco/
VK_LAYER_PATH=
ROS_PYTHON_VERSION=3
__GL_0x542675=0x4c0
EXP_PATH=/mnt/dev-tools/isaac-sim-5.1/apps
HOSTNAME=jupyter-yxzhan-2disaacsim-2dtemplate-2ddj0wlqco
LANGUAGE=C.UTF-8
HDST_USE_TRANSLUCENT_MATERIAL_TAG=1
JUPYTERHUB_API_TOKEN=76548cde850e4467995db7ac905ff1fc
PROXY_API_SERVICE_PORT=8001
GZ_CONFIG_PATH=/opt/ros/jazzy/opt/gz_sim_vendor/share/gz:/opt/ros/jazzy/opt/sdformat_vendor/share/gz:/opt/ros/jazzy/opt/gz_gui_vendor/share/gz:/opt/ros/jazzy/opt/gz_transport_vendor/share/gz:/opt/ros/jazzy/opt/gz_rendering_vendor/share/gz:/opt/ros/jazzy/opt/gz_plugin_vendor/share/gz:/o

In [13]:
from isaacsim.core.utils.extensions import enable_extension

enable_extension("isaacsim.ros2.bridge")

[9.398s] Simulation App Startup Complete
2025-11-04T13:13:41Z [18,147ms] [Warning] [omni.fabric.plugin] getAttributeCount called on non-existent path /World/Apartment/tap_handle/collisions/IAIKitchenWaterTapHandle
2025-11-04T13:13:41Z [18,147ms] [Warning] [omni.fabric.plugin] getTypes called on non-existent path /World/Apartment/tap_handle/collisions/IAIKitchenWaterTapHandle
2025-11-04T13:13:46Z [23,331ms] [Warning] [omni.physx.plugin] Detected an articulation at /World/Anymal/base with more than 4 velocity iterations being added to a TGS scene.The related behavior changed recently, please consult the changelog. This warning will only print once.
2025-11-04T13:13:46Z [23,509ms] [Warning] [omni.hydra] Mesh '/__Prototype_4712143806621822611/mesh_0' has corrupted data in primvar 'st': buffer size 702 doesn't match expected size 12828 in faceVarying primvars
2025-11-04T13:13:49Z [26,567ms] [Warning] [carb] Client gpu.foundation.plugin has acquired [gpu::unstable::IMemoryBudgetManagerFactor

True

## Create a ROS node listen to topic `cmd_vel`

In [14]:
import rclpy
from geometry_msgs.msg import Twist

if not rclpy.ok():
    rclpy.init(args=None)



    
# node = rclpy.create_node("carter_stereo")
# publisher = node.create_publisher(Twist, "cmd_vel", 10)
# message = Twist()
# message.angular.z = 0.5  # spin in place
# publisher.publish(message)
# node.destroy_node()
# rclpy.shutdown()

ModuleNotFoundError: No module named 'rclpy._rclpy_pybind11'
The C extension '/opt/ros/jazzy/lib/python3.12/site-packages/_rclpy_pybind11.cpython-311-x86_64-linux-gnu.so' isn't present on the system. Please refer to 'https://docs.ros.org/en/jazzy/How-To-Guides/Installation-Troubleshooting.html#import-failing-without-library-present-on-the-system' for possible solutions

## Running the simulation continuously

Once the following code cell is executed, it will enter an infinite loop. You can only terminate the entire program by restarting the kernel, the "Shutdown" button below is a shortcut, after which you'll need to rerun the previous code.

<button data-commandlinker-command="notebook:restart-clear-output" class="jupyter-button">Shutdown</button>

In [ ]:
# Reset Status
first_step = True
my_world.reset()

# A series of commands.
plans = [
    [0.3, 0.0, 0.0],
    [0.0, 0.0, 0.8],
    [0.3, 0.0, 0.0],
    [-0.3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
]

commands = plans.pop(0)

# Each command is executed for N simulation frames.
N = 100
frame_count = 0

# while len(plans) != 0:
while simulation_app.is_running():
    my_world.step(render=True)
    # when stop in UI
    if my_world.is_stopped():
        first_step = True
        commands = [0.0, 0.0, 0.0]
    if my_world.is_playing() and len(plans) != 0:
        frame_count += 1
        if frame_count % N == 0:
            commands = plans.pop(0)